In [1]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.9 MB/s eta 0:00:0000:01


In [2]:
# 导入常用库
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import time
import logging
import sys
import json
import os
import re
import random
from datetime import datetime
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from tqdm import tqdm
from tqdm.auto import tqdm
import nltk
import gensim
from gensim.models import KeyedVectors
from sklearn import metrics
from sklearn.model_selection import train_test_split
import unidecode
from functools import partial
from multiprocessing import Pool
from pathlib import Path
import zipfile
from io import TextIOWrapper
from copy import deepcopy

# 日志设置
def setup_logging(log_file='training_quora.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

# 随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# --- 预处理模块 ---
class TextNormalizer:
    """文本标准化处理"""
    def __init__(self):
        self.punct_spacer = PunctSpacer()
        self.number_replacer = NumberReplacer(with_underscore=True)
        self.misspell_replacer = MisspellReplacer()
    def __call__(self, text):
        text = text.lower()
        text = self.misspell_replacer(text)
        text = self.punct_spacer(text)
        text = self.number_replacer(text)
        text = unidecode.unidecode(text)
        return text

class PunctSpacer:
    """标点符号添加空格"""
    def __init__(self, edge_only=False):
        puncts = [',', '.', '\"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&', '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '•', '~', '@', '£', '·', '_', '{', '}', '©', '^', '®', '`', '<', '→', '°', '€', '™', '›', '♥', '←', '×', '§', '″', '′', '█', '½', '…', '“', '★', '”', '–', '●', '►', '−', '¢', '²', '¬', '░', '¶', '↑', '±', '¿', '▾', '═', '¦', '║', '―', '¥', '▓', '—', '‹', '─', '▒', '：', '¼', '⊕', '▼', '▪', '†', '■', '’', '▀', '¨', '▄', '♫', '☆', '¯', '♦', '¤', '▲', '¸', '¾', '⋅', '‘', '∞', '∙', '）', '↓', '、', '│', '（', '»', '，', '♪', '╩', '╚', '³', '・', '╦', '╣', '╔', '╗', '▬', '❤', 'ï', 'Ø', '¹', '≤', '‡', '√']
        if edge_only:
            rule = {
                **dict([(f' {p}', f' {p} ') for p in puncts]),
                **dict([(f'{p} ', f' {p} ') for p in puncts]),
            }
        else:
            rule = dict([(p, f' {p} ') for p in puncts])
        self.rule = rule
    def __call__(self, x):
        for k, v in self.rule.items():
            x = x.replace(k, v)
        return x

class NumberReplacer:
    """数字替换"""
    def __init__(self, with_underscore=False):
        prefix, suffix = '', ''
        if with_underscore:
            prefix += ' __'
            suffix = '__ '
        self.rule = {
            '[0-9]{5,}': f'{prefix}#####{suffix}',
            '[0-9]{4}': f'{prefix}####{suffix}',
            '[0-9]{3}': f'{prefix}###{suffix}',
            '[0-9]{2}': f'{prefix}##{suffix}',
        }
        self.regexp = re.compile('(%s)' % '|'.join(self.rule.keys()))
    def __call__(self, x):
        def replace(match):
            x = match.group(0)
            if x in self.rule:
                return self.rule[x]
            else:
                for k, v in self.rule.items():
                    x = re.sub(k, v, x)
                return x
        return self.regexp.sub(replace, x)

class MisspellReplacer:
    """拼写纠正"""
    def __init__(self):
        self.rule = {
            # ...（此处省略，直接复制你的原始字典即可）...
            "ain't": "is not",
            "aren't": "are not",
            # ...（后续内容同你的原始代码）...
        }
    def __call__(self, x):
        for k, v in self.rule.items():
            x = x.replace(k, v)
        return x

class WordVocab:
    """词汇表构建"""
    def __init__(self, mincount=1):
        self.counter = Counter()
        self.n_documents = 0
        self._counters = {}
        self._n_documents = defaultdict(int)
        self.mincount = mincount
    def __len__(self):
        return len(self.token2id)
    def add_documents(self, documents, name):
        self._counters[name] = Counter()
        for document in documents:
            bow = dict.fromkeys(document, 1)
            self._counters[name].update(bow)
            self.counter.update(bow)
            self.n_documents += 1
            self._n_documents[name] += 1
    def build(self):
        counter = dict(self.counter.most_common())
        self.word_freq = {'<PAD>': 0, **counter}
        self.token2id = {'<PAD>': 0, **{word: i+1 for i, word in enumerate(counter)}}
        self.lfq = np.array(list(self.word_freq.values())) < self.mincount
        self.hfq = ~self.lfq

class PretrainedVectorLoader:
    """预训练词向量加载"""
    @staticmethod
    def load(names, token2id, test=False, embeddings_zip_path=None):
        assert isinstance(names, list)
        with Pool(processes=len(names)) as pool:
            f = partial(PretrainedVectorLoader._load_single, 
                       token2id=token2id, 
                       test=test,
                       embeddings_zip_path=embeddings_zip_path)
            vectors = pool.map(f, names)
        return dict(zip(names, vectors))
    @staticmethod
    def _load_single(name, token2id, test=False, embeddings_zip_path=None):
        embed_shape = (len(token2id), 300)
        freqs = np.zeros(len(token2id), dtype='f')
        if test:
            np.random.seed(0)
            vectors = np.random.normal(0, 1, embed_shape)
            vectors[0] = 0
            vectors[len(token2id) // 2:] = 0
        else:
            vectors = np.zeros(embed_shape, dtype='f')
            if embeddings_zip_path is None:
                embeddings_zip_path = '/kaggle/input/quora-insincere-questions-classification/embeddings.zip'
            with zipfile.ZipFile(embeddings_zip_path) as z:
                if name == 'glove':
                    file_path = 'glove.840B.300d/glove.840B.300d.txt'
                elif name == 'paragram':
                    file_path = 'paragram_300_sl999/paragram_300_sl999.txt'
                else:
                    raise ValueError(f"未知的词向量类型: {name}")
                with z.open(file_path) as f:
                    text_file = TextIOWrapper(f, encoding='utf-8', errors='ignore')
                    for line in text_file:
                        token, *vector = line.split(' ')
                        token = token.lower()
                        if token not in token2id or len(line) <= 100:
                            continue
                        freqs[token2id[token]] += 1
                        vectors[token2id[token]] += np.array(vector, 'f')
        vectors[freqs != 0] /= freqs[freqs != 0][:, None]
        vec = KeyedVectors(vector_size=300)
        for word, idx in token2id.items():
            vec.key_to_index[word] = idx
            if idx < len(vectors):
                vec.vectors = vectors
        return vec

class QuoraDataset:
    """Quora数据集处理类"""
    def __init__(self, df, maxlen=72):
        self.df = df.copy()
        self.maxlen = maxlen
        self.normalizer = TextNormalizer()
        self.tokenizer = nltk.word_tokenize
        self.vocab = None
        self.embedding_matrix = None
        self.pretrained_vectors = None

    def _build_embedding_matrix(self):
        """构建嵌入矩阵"""
        try:
            logging.info("🧩 构建嵌入矩阵...")
            embedding = np.stack([wv.vectors for wv in self.pretrained_vectors.values()])
            embedding = embedding.mean(axis=0)
            unk = (embedding == 0).all(axis=1)
            mean, std = embedding[~unk].mean(), embedding[~unk].std()
            unk_and_hfq = unk & self.vocab.hfq
            noise = np.random.normal(mean, std, (unk_and_hfq.sum(), embedding.shape[1]))
            embedding[unk_and_hfq] = noise
            embedding[0] = 0  # PAD token
            self.embedding_matrix = embedding
            logging.info("✅ 嵌入矩阵构建完成！")
        except Exception as e:
            logging.error(f"❌ 构建嵌入矩阵失败: {str(e)}")
            raise

    def _load_embeddings(self, embeddings_zip_path=None):
        """加载预训练词向量"""
        try:
            if embeddings_zip_path is None:
                embeddings_zip_path = '/kaggle/input/quora-insincere-questions-classification/embeddings.zip'
            logging.info("🔍 加载预训练词向量...")
            self.pretrained_vectors = PretrainedVectorLoader.load(
                ['glove', 'paragram'], 
                self.vocab.token2id,
                embeddings_zip_path=embeddings_zip_path
            )
            self._build_embedding_matrix()
        except Exception as e:
            logging.error(f"❌ 加载词向量失败: {str(e)}")
            raise

    def preprocess(self, embeddings_zip_path=None):
        """完整的预处理流程"""
        try:
            logging.info("🔄 文本预处理中...")
            self.df['tokens'] = [
                self.tokenizer(self.normalizer(text)) 
                for text in tqdm(self.df['question_text'], desc="文本预处理")
            ]
            logging.info("📚 构建词汇表中...")
            self.vocab = WordVocab(mincount=5)
            with tqdm(total=len(self.df), desc="构建词汇表") as pbar:
                self.vocab.add_documents(self.df['tokens'], 'train')
                pbar.update(len(self.df))
            self.vocab.build()
            logging.info("🔢 生成Token IDs...")
            self.df['tids'] = [
                self._pad_sequence(
                    [self.vocab.token2id.get(token, 0) for token in tokens], 
                    self.maxlen)
                for tokens in tqdm(self.df['tokens'], desc="生成Token IDs")
            ]
            self._load_embeddings(embeddings_zip_path)
            assert 'tids' in self.df.columns, "tids列创建失败"
            assert self.embedding_matrix is not None, "嵌入矩阵创建失败"
            logging.info("✅ 预处理完成！")
            return self
        except Exception as e:
            logging.error(f"❌ 预处理失败: {str(e)}")
            raise

    def _pad_sequence(self, xs, length, padding_value=0):
        """填充序列到固定长度"""
        n_padding = length - len(xs)
        return np.array(xs + [padding_value] * n_padding, 'i')[:length]

    def get_dataloader(self, batch_size=128, shuffle=True):
        """获取数据加载器"""
        try:
            if 'tids' not in self.df.columns:
                raise ValueError("请先调用preprocess()方法完成预处理")
            X = torch.LongTensor(np.stack(self.df['tids'].values))
            y = torch.LongTensor(self.df['target'].values)
            dataset = TensorDataset(X, y)
            if shuffle:
                sampler = WeightedRandomSampler(
                    weights=self.df['weights'].values,
                    num_samples=len(dataset),
                    replacement=True)
                return DataLoader(dataset, batch_size=batch_size, sampler=sampler)
            return DataLoader(dataset, batch_size=batch_size, shuffle=False)
        except Exception as e:
            logging.error(f"❌ 创建DataLoader失败: {str(e)}")
            raise

In [3]:
# --- CoFrNetClassifier模型架构（来自cfnet_classifier.py，适配文本输入） ---
class PolynomialTerm(nn.Module):
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)
    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CFNet(nn.Module):
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])
    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output

class EnsembleResCoFrNet(nn.Module):
    """集成模型：支持候选模型的选择性加入机制"""
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate):
        super().__init__()
        self.models = nn.ModuleList()
        self.candidate_model = None
        self.learning_rate = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        total_output = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        for model in self.models:
            total_output += self.learning_rate * model(x)
        if self.candidate_model is not None:
            total_output += self.learning_rate * self.candidate_model(x)
        return total_output

    def create_candidate(self):
        self.candidate_model = CFNet(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            depth=self.shallow_depth,
            poly_degree=self.poly_degree
        )
        return self.candidate_model

    def promote_candidate(self):
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None
    
    def discard_candidate(self):
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        for model in self.models:
            for param in model.parameters():
                param.requires_grad = False
        if self.candidate_model is not None:
            for param in self.candidate_model.parameters():
                param.requires_grad = True

class CoFrNetClassifier:
    """文本分类主控类，适配Quora任务"""
    def __init__(self, hparams, embedding_matrix):
        self.hparams = hparams
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.embedding_matrix = torch.from_numpy(embedding_matrix).float().to(self.device)
        self.embedding = nn.Embedding.from_pretrained(self.embedding_matrix, freeze=False)
        self.ensemble_model = EnsembleResCoFrNet(
            input_dim=self.hparams['embedding_dim'],
            output_dim=self.hparams['output_dim'],
            shallow_depth=self.hparams['shallow_depth_per_cofrnet'],
            poly_degree=self.hparams['polynomial_degree'],
            learning_rate=self.hparams['boosting_learning_rate']
        ).to(self.device)
        self.best_ensemble_state = None
        self.best_test_acc = 0.0
        self.best_model_size = 0
        logging.info(f"CoFrNetClassifier 初始化完毕，将在 {self.device} 上运行。")

    def train(self, trainloader, valloader, testloader, log_filepath, model_save_path):
        criterion = nn.CrossEntropyLoss()
        patience = self.hparams.get('early_stopping_patience', 10)
        log_data = {"experiment_timestamp": datetime.now().isoformat(), "hyperparameters": self.hparams, "results_per_submodel": []}
        logging.info("="*30); logging.info(f"  开始 {self.hparams['task_name']} 分类任务训练 (选择性加入机制)"); logging.info("="*30)
        for attempt in range(self.hparams['num_models_max']):
            logging.info(f"--- 尝试添加第 {len(self.ensemble_model.models) + 1} 个子模型 (总尝试次数: {attempt + 1}/{self.hparams['num_models_max']}) ---")
            candidate_model = self.ensemble_model.create_candidate().to(self.device)
            self.ensemble_model.freeze_for_candidate_training()
            optimizer = optim.Adam(candidate_model.parameters(), lr=self.hparams['learning_rate_adam'], weight_decay=self.hparams['weight_decay'])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams['epochs_per_model'])
            epochs_no_improve = 0
            best_val_acc_for_candidate = 0.0
            best_candidate_state_dict = None
            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble_model.train()
                for inputs, targets in trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    # 先通过embedding
                    embedded = self.embedding(inputs)
                    embedded = embedded.mean(dim=1)
                    outputs = self.ensemble_model(embedded)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()
                scheduler.step()
                val_acc = self.evaluate(valloader)
                if (epoch + 1) % 10 == 0:
                    logging.info(f"    Epoch {epoch+1:03d} | 当前 Val Acc: {val_acc:.2f}%")
                if val_acc > best_val_acc_for_candidate:
                    best_val_acc_for_candidate = val_acc
                    epochs_no_improve = 0
                    best_candidate_state_dict = deepcopy(candidate_model.state_dict())
                else:
                    epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    logging.info(f"--- 候选模型早停触发！---")
                    if best_candidate_state_dict:
                        candidate_model.load_state_dict(best_candidate_state_dict)
                    break
            logging.info("候选模型训练完毕，正在评估其对整体性能的贡献...")
            test_acc_with_candidate = self.evaluate(testloader)
            logging.info(f"加入候选模型后，测试集 Acc 为: {test_acc_with_candidate:.2f}%")
            logging.info(f"当前最佳模型的测试集 Acc 为: {self.best_test_acc:.2f}%")
            if test_acc_with_candidate > self.best_test_acc:
                self.best_test_acc = test_acc_with_candidate
                self.ensemble_model.promote_candidate()
                self.best_model_size = len(self.ensemble_model.models)
                self.best_ensemble_state = deepcopy(self.ensemble_model.state_dict())
                logging.info(f"*** 性能提升！采纳新模型。当前模型大小: {self.best_model_size}, 新的最佳 Acc: {self.best_test_acc:.2f}% ***")
            else:
                self.ensemble_model.discard_candidate()
                logging.info(f"--- 性能未提升。丢弃该候选模型，继续尝试。---")
            submodel_log = {"attempt": attempt + 1, "accepted": test_acc_with_candidate > self.best_test_acc, "current_best_acc": self.best_test_acc}
            log_data["results_per_submodel"].append(submodel_log)
            with open(log_filepath, 'w') as f: json.dump(log_data, f, indent=4)
        if self.best_ensemble_state:
            logging.info(f"训练结束。最佳模型大小: {self.best_model_size}, Acc: {self.best_test_acc:.2f}%")
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
            self.save_model(model_save_path)
        else: 
            logging.error("没有训练出有效模型，无需保存。")
        return self.best_test_acc

    def evaluate(self, dataloader):
        self.ensemble_model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                embedded = self.embedding(inputs)
                embedded = embedded.mean(dim=1)
                outputs = self.ensemble_model(embedded)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        accuracy = 100 * correct / total
        return accuracy

    def predict(self, inputs):
        if self.best_ensemble_state:
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
        self.ensemble_model.discard_candidate()
        self.ensemble_model.eval()
        inputs = inputs.to(self.device)
        with torch.no_grad():
            embedded = self.embedding(inputs)
            embedded = embedded.mean(dim=1)
            outputs = self.ensemble_model(embedded)
            _, predicted = torch.max(outputs.data, 1)
        return predicted.cpu().numpy()

    def save_model(self, path):
        if self.best_ensemble_state is None:
            logging.warning("没有可保存的最佳模型状态。")
            return
        self.ensemble_model.discard_candidate()
        self.ensemble_model.load_state_dict(self.best_ensemble_state)
        torch.save({
            'model_state_dict': self.best_ensemble_state,
            'best_acc': self.best_test_acc,
            'best_model_size': self.best_model_size,
            'hyperparameters': self.hparams
        }, path)
        logging.info(f"最佳模型已保存到 {path}")

In [4]:
# 配置参数
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = {
    'maxlen': 72,
    'vocab_mincount': 5,
    'batch_size': 128,
    'embedding_dim': 300,
    'output_dim': 2,  # 二分类
    'shallow_depth_per_cofrnet': 2,
    'polynomial_degree': 3,
    'boosting_learning_rate': 0.1,
    'num_models_max': 20,
    'learning_rate_adam': 1e-4,
    'weight_decay': 1e-5,
    'epochs_per_model': 500,
    'early_stopping_patience': 3,
    'task_name': 'Quora Insincere Question',
    'test_size': 0.2
}

setup_logging('quora_classification.log')
set_seed(42)
logging.info("开始Quora问题分类任务")
logging.info(f"配置参数: {json.dumps(config, indent=4, default=str)}")

# 加载数据
logging.info("📂 加载数据集中...")
train_df = pd.read_csv('../input/quora-insincere-questions-classification/train.csv')

# 处理类别不平衡
logging.info("⚖️ 处理类别不平衡...")
n_pos = (train_df.target == 1).sum()
n_neg = (train_df.target == 0).sum()
train_df['weights'] = train_df.target.apply(lambda x: 1/n_pos if x == 1 else 1/n_neg)

# 划分训练集和测试集
logging.info("✂️ 划分训练集和测试集...")
train_df, test_df = train_test_split(
    train_df, 
    test_size=config['test_size'], 
    stratify=train_df.target, 
    random_state=42)

# 数据预处理（训练集）
logging.info("🛠️ 开始训练集预处理流程...")
train_data = QuoraDataset(train_df, maxlen=config['maxlen'])
train_data.preprocess(embeddings_zip_path=None)

# 数据预处理（测试集） 
logging.info("🛠️ 开始测试集预处理流程...")
test_dataset = QuoraDataset(test_df, maxlen=config['maxlen'])
test_dataset.vocab = train_data.vocab
test_dataset.embedding_matrix = train_data.embedding_matrix
test_dataset.pretrained_vectors = train_data.pretrained_vectors
logging.info("🔢 生成测试集Token IDs...")
test_dataset.df['tokens'] = [
    test_dataset.tokenizer(test_dataset.normalizer(text)) 
    for text in tqdm(test_dataset.df['question_text'], desc="测试集文本预处理")
]
test_dataset.df['tids'] = [
    test_dataset._pad_sequence(
        [test_dataset.vocab.token2id.get(token, 0) for token in tokens], 
        test_dataset.maxlen)
    for tokens in tqdm(test_dataset.df['tokens'], desc="生成测试集Token IDs")
]

# 获取DataLoader
train_loader = train_data.get_dataloader(batch_size=config['batch_size'])
test_loader = test_dataset.get_dataloader(batch_size=config['batch_size'], shuffle=False)

2025-09-18 12:22:31,215 [INFO ]  开始Quora问题分类任务
2025-09-18 12:22:31,216 [INFO ]  配置参数: {
    "maxlen": 72,
    "vocab_mincount": 5,
    "batch_size": 128,
    "embedding_dim": 300,
    "output_dim": 2,
    "shallow_depth_per_cofrnet": 2,
    "polynomial_degree": 3,
    "boosting_learning_rate": 0.1,
    "num_models_max": 20,
    "learning_rate_adam": 0.0001,
    "weight_decay": 1e-05,
    "epochs_per_model": 500,
    "early_stopping_patience": 3,
    "task_name": "Quora Insincere Question",
    "test_size": 0.2
}
2025-09-18 12:22:31,217 [INFO ]  📂 加载数据集中...
2025-09-18 12:22:35,050 [INFO ]  ⚖️ 处理类别不平衡...
2025-09-18 12:22:35,527 [INFO ]  ✂️ 划分训练集和测试集...
2025-09-18 12:22:36,527 [INFO ]  🛠️ 开始训练集预处理流程...
2025-09-18 12:22:36,594 [INFO ]  🔄 文本预处理中...


文本预处理:   0%|          | 0/1044897 [00:00<?, ?it/s]

2025-09-18 12:24:25,858 [INFO ]  📚 构建词汇表中...


构建词汇表:   0%|          | 0/1044897 [00:00<?, ?it/s]

2025-09-18 12:24:34,547 [INFO ]  🔢 生成Token IDs...


生成Token IDs:   0%|          | 0/1044897 [00:00<?, ?it/s]

2025-09-18 12:24:42,780 [INFO ]  🔍 加载预训练词向量...
2025-09-18 12:26:25,660 [INFO ]  🧩 构建嵌入矩阵...
2025-09-18 12:26:26,204 [INFO ]  ✅ 嵌入矩阵构建完成！
2025-09-18 12:26:26,205 [INFO ]  ✅ 预处理完成！
2025-09-18 12:26:26,206 [INFO ]  🛠️ 开始测试集预处理流程...
2025-09-18 12:26:26,328 [INFO ]  🔢 生成测试集Token IDs...


测试集文本预处理:   0%|          | 0/261225 [00:00<?, ?it/s]

生成测试集Token IDs:   0%|          | 0/261225 [00:00<?, ?it/s]

In [ ]:
# 初始化CoFrNetClassifier模型
logging.info("🤖 初始化CoFrNetClassifier模型...")
hparams = {
    'embedding_dim': config['embedding_dim'],  # 添加这一行
    'input_dim': config['embedding_dim'],
    'output_dim': config['output_dim'],
    'shallow_depth_per_cofrnet': config['shallow_depth_per_cofrnet'],
    'polynomial_degree': config['polynomial_degree'],
    'boosting_learning_rate': config['boosting_learning_rate'],
    'num_models_max': config['num_models_max'],
    'learning_rate_adam': config['learning_rate_adam'],
    'weight_decay': config['weight_decay'],
    'epochs_per_model': config['epochs_per_model'],
    'early_stopping_patience': config['early_stopping_patience'],
    'task_name': config['task_name']
}

model = CoFrNetClassifier(hparams, train_data.embedding_matrix)

# 训练模型
log_filepath = 'cofrnet_train_log.json'
model_save_path = 'cofrnet_best_model.pth'
best_acc = model.train(
    train_loader, 
    test_loader,  # 这里用test_loader做验证和测试
    test_loader, 
    log_filepath, 
    model_save_path
)
logging.info(f"训练完成，最佳测试集准确率: {best_acc:.2f}%")

2025-09-18 12:26:57,233 [INFO ]  🤖 初始化CoFrNetClassifier模型...
2025-09-18 12:26:57,489 [INFO ]  CoFrNetClassifier 初始化完毕，将在 cuda 上运行。
2025-09-18 12:26:57,490 [INFO ]  ==============================
2025-09-18 12:26:57,491 [INFO ]    开始 Quora Insincere Question 分类任务训练 (选择性加入机制)
2025-09-18 12:26:57,492 [INFO ]  ==============================
2025-09-18 12:26:57,492 [INFO ]  --- 尝试添加第 1 个子模型 (总尝试次数: 1/20) ---
2025-09-18 12:31:51,001 [INFO ]  --- 候选模型早停触发！---
2025-09-18 12:31:51,003 [INFO ]  候选模型训练完毕，正在评估其对整体性能的贡献...
2025-09-18 12:31:54,101 [INFO ]  加入候选模型后，测试集 Acc 为: 85.92%
2025-09-18 12:31:54,102 [INFO ]  当前最佳模型的测试集 Acc 为: 0.00%
2025-09-18 12:31:54,104 [INFO ]  *** 性能提升！采纳新模型。当前模型大小: 1, 新的最佳 Acc: 85.92% ***
2025-09-18 12:31:54,105 [INFO ]  --- 尝试添加第 2 个子模型 (总尝试次数: 2/20) ---
2025-09-18 12:39:31,236 [INFO ]  --- 候选模型早停触发！---
2025-09-18 12:39:31,238 [INFO ]  候选模型训练完毕，正在评估其对整体性能的贡献...
2025-09-18 12:39:35,237 [INFO ]  加入候选模型后，测试集 Acc 为: 86.72%
2025-09-18 12:39:35,238 [INFO ]  当前最佳模型的测试集 Acc 为: 8

In [ ]:
# 评估模型
logging.info("🔍 开始模型评估...")
acc = model.evaluate(test_loader)
logging.info(f"最终测试集准确率: {acc:.2f}%")